<a href="https://colab.research.google.com/github/GautamSutar/PEFT-Fine-Tuning-Pipeline/blob/main/Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
%pip install transformers
%pip install datasets
%pip install peft
%pip install trl
%pip install accelerate
%pip install bitsandbytes
%pip install torch

In [15]:
from datasets import load_dataset

In [16]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)

In [17]:
from peft import (
    LoraConfig,
    get_peft_model,
)

In [18]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

In [19]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": "data/train.json"}
)["train"]

In [20]:
import json

with open("data/train.json", "r") as f:
    data = json.load(f)

print(len(data))
print(data[0])

200
{'instruction': 'What is Python?', 'output': 'Python is a high-level, interpreted programming language known for readability and a rich ecosystem.'}


In [21]:
for row in dataset:
  print(row)

{'instruction': 'What is Python?', 'output': 'Python is a high-level, interpreted programming language known for readability and a rich ecosystem.'}
{'instruction': 'What is Django?', 'output': 'Django is a high-level Python web framework following the MVT architecture.'}
{'instruction': 'What is FastAPI?', 'output': 'FastAPI is a modern Python framework for building high-performance APIs.'}
{'instruction': 'What is Flask?', 'output': 'Flask is a lightweight Python web framework.'}
{'instruction': 'What is OOP?', 'output': 'Object-Oriented Programming organizes code into classes and objects.'}
{'instruction': 'What is List?', 'output': 'A list is a mutable ordered collection in Python.'}
{'instruction': 'What is Tuple?', 'output': 'A tuple is an immutable ordered collection in Python.'}
{'instruction': 'What is Dictionary?', 'output': 'A dictionary stores key-value pairs.'}
{'instruction': 'What is Set?', 'output': 'A set stores unique unordered elements.'}
{'instruction': 'What is Rec

In [22]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [23]:
def preprocess(example):

    text = f"""
### Instruction:
{example["instruction"]}

### Response:
{example["output"]}
"""

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=256,
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens


In [24]:
dataset = dataset.map(preprocess)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [25]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [26]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "v_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)


In [27]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [28]:
model = get_peft_model(model, config)

In [29]:
training_args = TrainingArguments(
    output_dir="./output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_steps=10,
    fp16=False,
)

In [30]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()

Step,Training Loss
1,13.982065
2,13.043214
3,7.598244
4,6.787731
5,5.304267
6,3.188892
7,1.862448
8,1.174666
9,0.590742
10,0.369531


TrainOutput(global_step=300, training_loss=0.2549289174253742, metrics={'train_runtime': 170.4924, 'train_samples_per_second': 3.519, 'train_steps_per_second': 1.76, 'total_flos': 330835466649600.0, 'train_loss': 0.2549289174253742, 'epoch': 3.0})

In [31]:
trainer.save_model("./fine_tuned_model")

In [32]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [33]:
import os

print(os.listdir("fine_tuned_model"))

['adapter_model.safetensors', 'adapter_config.json', 'README.md', 'training_args.bin']


In [34]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model = PeftModel.from_pretrained(
    base_model,
    "fine_tuned_model"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [35]:
prompt = """
### Instruction:
What is Python?

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True
)

print(tokenizer.decode(output[0], skip_special_tokens=True))


### Instruction:
What is Python?

### Response:
Python is a high-level, interpreted programming language known for its syntax and ecosystem.

